# Jalon 2 — inv-2-MR-Sort : explication des données

On cherche une partition des objets en **2 segments** de clients, et un modèle MR-Sort par segment. Chaque modèle est appris avec l'heuristique de `lincs`.

Le jalon 2 est un problème d'**explication**, pas de prédiction :
- on apprend sur **100 % des objets** de chaque fichier ; il n'y a ni jeu de test ni jeu de validation ;
- on mesure la part des objets **expliqués**, c'est-à-dire bien classés par au moins un des deux modèles ;
- on compare après coup la taille des segments trouvés aux proportions du nom de fichier.

Choisir le modèle d'un objet nouveau relève du jalon 3.

**Hypothèses.** Les features sont dans $[0, 1]$ et à maximiser. Les catégories vont de la pire à la meilleure. L'algorithme est **aveugle** : il ne voit ni `rep` ni `noise`, qui servent seulement à l'évaluation.

Le code est celui de `02_jalon2_final.ipynb` (avec `SHARE_AMBIGUOUS = False`). Le script `jalon2_explication.py` contient exactement les mêmes cellules ; il sert à lancer les calculs longs en arrière-plan.

In [ ]:
import argparse, re, time
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import lincs, lincs.classification as C

ROOT = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path("..").resolve()
DATA_DIR = ROOT / "Datasets-Step2"
OUT_CSV  = ROOT / "results" / "jalon2_explication.csv"

ITER_SECONDS    = 1.0     # temps lincs par apprentissage, pendant les itérations (pour 6 critères / 3 catégories)
POLISH_SECONDS  = 4.0     # temps lincs de l'apprentissage final (polish), même référence
MAX_ITER        = 20      # nombre maximal d'itérations par départ
INITS           = ("errors", "random", "random", "random")   # départs : 1 guidé + 3 aléatoires
MIN_FRAC        = 0.10    # taille minimale d'un segment pour apprendre son modèle
SHARE_AMBIGUOUS = False   # chaque objet appartient à un seul segment (choix du jalon 2)
print("lincs", lincs.__version__, "| données :", DATA_DIR)

## 1. Chargement

`problem.yml` est lu avec `yaml`, le CSV avec pandas. La classe `y` est l'indice de la catégorie, de 0 (la pire) à $p-1$. `file_info` lit `N`, `rep` et `noise` dans le nom du fichier, pour l'évaluation seulement.

In [ ]:
def load_dataset(csv_path):
    csv_path = Path(csv_path)
    pb = yaml.safe_load((csv_path.parent / "problem.yml").read_text())
    assert all(c.get("preference_direction", "increasing") == "increasing" for c in pb["criteria"])
    criteria = [str(c["name"]) for c in pb["criteria"]]
    categories = [str(c["name"]) for c in pb["ordered_categories"]]
    df = pd.read_csv(csv_path)
    y = df["category"].astype(str).map({c: h for h, c in enumerate(categories)})
    assert not y.isna().any(), "catégorie inconnue"
    return df[criteria].to_numpy(float), y.to_numpy(int), len(categories)

def file_info(csv_path):
    # rep et bruit servent UNIQUEMENT à l'évaluation après coup : l'algorithme ne les voit pas
    csv_path = Path(csv_path)
    info = dict(type=csv_path.parent.parent.name, dossier=csv_path.parent.name, fichier=csv_path.stem)
    m = re.search(r"n(\d+)_rep(\d+)-(\d+)_noise(\d+)", csv_path.name)
    if m is None:
        return dict(info, N=np.nan, rep="?", petit_attendu=np.nan, bruit=np.nan)
    return dict(info, N=int(m[1]), rep=f"{m[2]}-{m[3]}", petit_attendu=min(int(m[2]), int(m[3])) / 100,
                bruit=int(m[4]) / 100)

## 2. Briques MR-Sort autour de lincs

- `learn_mrsort` : heuristique *weights-profiles-breed* de lincs (9 modèles, les 4 moins bons réinitialisés à chaque itération). Arrêt à 100 % de précision, au bout du temps alloué, ou après 30 itérations sans progrès.
- `budget` : le temps alloué est proportionnel au nombre de paramètres, $n(p-1) + n + 1$.
- `predict` et `margins` appliquent la règle MR-Sort en numpy, en float32 comme lincs.

**Marge** d'un objet $x$ de catégorie $h$ : $m(x) = \min\big(C(x, b^{h-1}) - \lambda,\ \lambda - C(x, b^{h})\big)$, avec des poids normalisés. Elle est positive si $x$ est bien classé.

In [ ]:
def n_params(n, p):
    return n * (p - 1) + n + 1

def budget(base_seconds, n, p):
    return base_seconds * n_params(n, p) / n_params(6, 3)

def make_problem(n, p):
    crits = [C.Criterion(f"c{i + 1}", C.Criterion.RealValues(C.Criterion.PreferenceDirection.increasing, 0., 1.))
             for i in range(n)]
    return C.Problem(crits, [C.Category(f"cat{h}") for h in range(p)])

def to_alts(problem, X, y=None):
    return C.Alternatives(problem, [
        C.Alternative(f"a{k}", [C.Performance(C.Performance.Real(float(v))) for v in X[k]],
                      None if y is None else int(y[k])) for k in range(len(X))])

def learn_mrsort(X, y, p, seed=0, seconds=2.0, models=9, patience=30):
    problem = make_problem(X.shape[1], p)
    pre = C.PreprocessedLearningSet(problem, to_alts(problem, X, y))
    mbl = C.LearnMrsortByWeightsProfilesBreed.ModelsBeingLearned(pre, models, int(seed))
    init = C.InitializeProfilesForProbabilisticMaximalDiscriminationPowerPerCriterion(pre, mbl)
    weights = C.OptimizeWeightsUsingGlop(pre, mbl)
    profiles = C.ImproveProfilesWithAccuracyHeuristicOnCpu(pre, mbl)
    breed = C.ReinitializeLeastAccurate(mbl, init, weights, count=models // 2)
    term = C.TerminateWhenAny([C.TerminateAtAccuracy(mbl, len(X)), C.TerminateAfterSeconds(seconds),
                               C.TerminateAfterIterationsWithoutProgress(mbl, patience)])
    m = C.LearnMrsortByWeightsProfilesBreed(pre, mbl, init, weights, profiles, breed, term).perform()
    B = np.array([[np.inf if av.real_thresholds.thresholds[h] is None else av.real_thresholds.thresholds[h]
                   for av in m.accepted_values] for h in range(p - 1)])          # (p-1, n)
    W = np.array(m.sufficient_coalitions[0].weights.criterion_weights)           # seuil de majorité = 1 (lincs)
    return {"profiles": B, "raw_weights": W, "weights": W / W.sum(), "lmbda": 1. / W.sum(),
            "_lincs": m, "_problem": problem}

def concordance(model, X):
    # C[k, h] = somme des poids (bruts) des critères où x_k >= b^h, en float32 comme lincs
    X32 = np.asarray(X, dtype=np.float32)
    W32 = np.asarray(model["raw_weights"], dtype=np.float32)
    return np.stack([(X32 >= b.astype(np.float32)).astype(np.float32) @ W32 for b in model["profiles"]],
                    axis=1).astype(float)

def predict(model, X):
    return np.cumprod(concordance(model, X) >= 1., axis=1).sum(axis=1)       # nb de frontières franchies

def margins(model, X, y):
    Cn = (concordance(model, X) - 1.) / model["raw_weights"].sum()          # C(x, b^h) - lambda, poids normalisés
    m = np.full(len(X), np.inf)
    low = y >= 1                                                             # doit franchir b^{h-1}
    m[low] = np.minimum(m[low], Cn[low, y[low] - 1] + 1e-12)
    up = np.where(y <= Cn.shape[1] - 1)[0]                                   # ne doit pas franchir b^h
    m[up] = np.minimum(m[up], -Cn[up, y[up]] - 1e-12)
    m[np.isinf(m)] = 1.
    return m

## 3. Algorithme inv-2-MR-Sort

Alternance de type k-means (*clusterwise regression*, Späth 1979) :

```
pour chaque départ (1 guidé « errors » + 3 aléatoires) :
    partition initiale
    répéter au plus MAX_ITER fois :
        apprendre un modèle lincs par segment
        réaffecter chaque objet au modèle qui l'explique strictement mieux (écart de catégorie, puis marge)
        s'arrêter si aucun objet n'a changé de segment
    garder la meilleure itération, puis réapprendre chaque modèle plus longtemps (polish)
retenir le départ qui explique le plus d'objets
```

Un objet bien classé par les deux modèles (**ambigu**) n'est jamais déplacé : il reste dans son segment de départ. Les données ne permettent pas de l'attribuer, ce qui rend les proportions imprécises.

In [ ]:
def scores(models, X, y):
    E = np.stack([np.abs(predict(m, X) - y) for m in models], axis=1)     # (N, 2) écart de catégorie
    M = np.stack([margins(m, X, y) for m in models], axis=1)              # (N, 2) marge
    return E, M

def reassign(models, X, y, seg):
    E, M = scores(models, X, y)
    key = E - 1e-3 * np.clip(M, -1, 1)               # lexicographique : l'écart domine, la marge départage
    best = key.argmin(axis=1)
    rows = np.arange(len(X))
    move = key[rows, best] < key[rows, seg] - 1e-12  # strictement mieux expliqué ailleurs
    return np.where(move, best, seg), E

def fit_segments(X, y, p, seg, shared, rng, seconds):
    models = []
    for k in (0, 1):
        idx = np.where((seg == k) | shared)[0]
        size_min = int(max(10, MIN_FRAC * len(X)))
        if len(idx) < size_min:                       # segment trop petit : tirage aléatoire
            idx = rng.choice(len(X), size=size_min, replace=False)
        models.append(learn_mrsort(X[idx], y[idx], p, seed=int(rng.integers(1e9)), seconds=seconds))
    return models

def init_partition(X, y, p, how, rng, seconds):
    if how == "random":
        return rng.integers(0, 2, len(X))
    single = learn_mrsort(X, y, p, seed=int(rng.integers(1e9)), seconds=seconds)
    return (predict(single, X) != y).astype(int)      # "errors" : erreurs du modèle unique -> segment 1

def learn_2_mrsort(X, y, p, init="errors", seed=0, share_ambiguous=SHARE_AMBIGUOUS,
                   max_iter=MAX_ITER, iter_seconds=ITER_SECONDS, polish_seconds=POLISH_SECONDS):
    rng = np.random.default_rng(seed)
    n = X.shape[1]
    t_iter, t_polish = budget(iter_seconds, n, p), budget(polish_seconds, n, p)
    seg = init_partition(X, y, p, init, rng, t_iter)
    shared = np.zeros(len(X), bool)
    best, history = None, []

    for it in range(max_iter):
        models = fit_segments(X, y, p, seg, shared, rng, t_iter)          # apprentissage
        new_seg, E = reassign(models, X, y, seg)                           # réaffectation
        score = (E.min(axis=1) == 0).mean()                                # expliqués par au moins un modèle
        sizes = np.bincount(new_seg, minlength=2)
        history.append(dict(iteration=it + 1, expliques=score, petit_segment=sizes.min() / sizes.sum(),
                            ambigus=(E == 0).all(axis=1).mean(), changements=(new_seg != seg).mean()))
        if best is None or score > best["score"]:
            best = dict(score=score, models=models, seg=new_seg.copy(), E=E)
        if (new_seg == seg).all():                                         # arrêt : partition stable
            break
        seg = new_seg
        if share_ambiguous:
            shared = (E == 0).all(axis=1)

    # polish : apprentissage plus long sur la partition retenue
    seg = best["seg"]
    shared = (best["E"] == 0).all(axis=1) if share_ambiguous else np.zeros(len(X), bool)
    polished = []
    for k in (0, 1):
        idx = np.where((seg == k) | shared)[0]
        old = best["models"][k]
        if len(idx) < 10:
            polished.append(old)
            continue
        new = learn_mrsort(X[idx], y[idx], p, seed=int(rng.integers(1e9)), seconds=t_polish)
        better = (predict(new, X[idx]) == y[idx]).mean() >= (predict(old, X[idx]) == y[idx]).mean()
        polished.append(new if better else old)
    new_seg, E = reassign(polished, X, y, seg)
    score = (E.min(axis=1) == 0).mean()
    if score >= best["score"]:
        best.update(score=score, models=polished, seg=new_seg, E=E)

    best["history"] = pd.DataFrame(history)
    best["init"] = init
    return best

def learn_2_mrsort_multistart(X, y, p, inits=INITS, seed=0, **kwargs):
    runs = [learn_2_mrsort(X, y, p, init=how, seed=seed * 100 + r, **kwargs) for r, how in enumerate(inits)]
    return max(runs, key=lambda r: r["score"]), runs

## 4. Mesures, sur 100 % des objets

Une ligne par (fichier, graine) :

| Colonne | Définition |
|---|---|
| `unique_expl` | part des objets bien classés par **un seul** MR-Sort appris sur tout le fichier (temps du polish) |
| `aleatoire_expl` | part des objets bien classés par au moins un de **deux** MR-Sort appris sur une partition **aléatoire** 50/50 (temps du polish, sans réaffectation) |
| `deux_expl` | part des objets bien classés par au moins un des deux modèles de l'algorithme |
| `ambigus` | part des objets bien classés par les **deux** modèles de l'algorithme |
| `petit_segment` | part du plus petit segment trouvé |
| `erreur_proportion` | $\lvert$`petit_segment − petit_attendu`$\rvert$ |
| `depart`, `temps` | départ retenu et durée de l'algorithme (hors références) |

**Lecture.** `aleatoire_expl − unique_expl` mesure l'effet « deux fois plus de paramètres, et on prend pour chaque objet le modèle qui le classe bien ». `deux_expl − aleatoire_expl` mesure l'apport de la segmentation elle-même.

In [ ]:
def explained(models, X, y):
    # (N, nb modèles) : True si le modèle classe bien l'objet
    return np.stack([predict(m, X) == y for m in models], axis=1)

def explain_file(csv, seed):
    """Une ligne de mesures pour (fichier, graine), sur 100 % des objets."""
    X, y, p = load_dataset(csv)
    rng = np.random.default_rng(seed)
    t_ref = budget(POLISH_SECONDS, X.shape[1], p)

    # référence 1 : un seul MR-Sort sur tout le fichier
    single = learn_mrsort(X, y, p, seed=seed, seconds=t_ref)
    # référence 2 : deux MR-Sort sur une partition aléatoire 50/50, sans réaffectation
    halves = rng.permutation(len(X)) % 2
    rand_models = [learn_mrsort(X[halves == k], y[halves == k], p, seed=seed + 1 + k, seconds=t_ref)
                   for k in (0, 1)]
    ok_rand = explained(rand_models, X, y)

    t = time.time()
    best, _ = learn_2_mrsort_multistart(X, y, p, seed=seed)
    temps = time.time() - t
    ok = explained(best["models"], X, y)
    sizes = np.bincount(best["seg"], minlength=2)
    info = file_info(csv)
    petit = sizes.min() / sizes.sum()
    return dict(**info, graine=seed,
                unique_expl=explained([single], X, y)[:, 0].mean(),
                aleatoire_expl=ok_rand.any(axis=1).mean(),
                deux_expl=ok.any(axis=1).mean(),
                ambigus=ok.all(axis=1).mean(),
                petit_segment=petit,
                erreur_proportion=abs(petit - info["petit_attendu"]),
                depart=best["init"], temps=temps)

## 5. Résultats

Le calcul prend de 1 à 10 minutes par fichier. Par défaut on relit `results/jalon2_explication.csv`, écrit par le script :

```
python3 notebooks/jalon2_explication.py --types type_a type_b --seeds 0
```

Passer `RUN = True` pour calculer ici les lignes manquantes.

In [ ]:
RUN = False
TYPES, SEEDS = ["type_a", "type_b"], [0]

if RUN:
    done = set()
    if OUT_CSV.exists():
        old = pd.read_csv(OUT_CSV)
        done = set(zip(old.type, old.dossier, old.fichier, old.graine))
    for f in [f for t in TYPES for f in sorted((DATA_DIR / t).glob("cr*_ca*/*.csv"))]:
        for s in SEEDS:
            i = file_info(f)
            if (i["type"], i["dossier"], i["fichier"], s) not in done:
                row = explain_file(f, s)
                pd.DataFrame([row]).to_csv(OUT_CSV, mode="a", header=not OUT_CSV.exists(), index=False)
                print(i["fichier"], s, f"deux = {row['deux_expl']:.3f}", flush=True)

res = pd.read_csv(OUT_CSV)
print(f"{len(res)} lignes : " + ", ".join(f"{t} {n} fichiers" for t, n in res.groupby("type").fichier.size().items())
      + f" | temps moyen {res.temps.mean() / 60:.1f} min par fichier")
EXPL = ["unique_expl", "aleatoire_expl", "deux_expl", "ambigus"]

### 5.1 Part expliquée, par dossier (critères / catégories) et bruit

In [ ]:
def table(df, by, cols):
    g = df.groupby(by)
    return g[cols].mean().round(3).assign(fichiers=g.size())

t1 = table(res, ["type", "dossier", "bruit"], EXPL)
t1 = pd.concat([t1, table(res.assign(dossier="tous"), ["type", "dossier", "bruit"], EXPL)])
t1

### 5.2 Proportions des segments, par répartition et bruit

In [ ]:
t2 = res.groupby(["type", "rep", "bruit"]).agg(
    petit_attendu=("petit_attendu", "first"), petit_segment=("petit_segment", "mean"),
    petit_segment_std=("petit_segment", "std"), erreur_proportion=("erreur_proportion", "mean"),
    ambigus=("ambigus", "mean"), fichiers=("fichier", "size")).round(3)
t2

### 5.3 Part expliquée, par taille du fichier et bruit

In [ ]:
t3 = table(res, ["type", "N", "bruit"], EXPL)
t3

## 6. Comparaison avec les anciens chiffres

- `results/bench_complet.csv` (notebook 02, graine 2, 31 fichiers `type_a`) : l'algorithme apprenait sur 90 % des objets ; `deux_train` est la part expliquée sur ces 90 %, `unique_train` celle d'un modèle unique.
- `results/resultats_jalon2.csv` (une version plus ancienne, 2 fichiers `cr06_ca03` de 300 objets) : `two_train` et `single_train`, mêmes définitions.

On compare fichier par fichier, sur les fichiers communs.

In [ ]:
old = pd.read_csv(ROOT / "results/bench_complet.csv")
cmp = res.merge(old[["type", "dossier", "fichier", "unique_train", "deux_train", "ambigus"]],
                on=["type", "dossier", "fichier"], suffixes=("", "_ancien"))
cmp["diff"] = cmp.deux_expl - cmp.deux_train
print(f"{len(cmp)} fichiers communs avec bench_complet.csv")
display(cmp.groupby("bruit")[["unique_train", "unique_expl", "deux_train", "deux_expl", "diff", "ambigus_ancien", "ambigus"]]
        .mean().round(3).assign(fichiers=cmp.groupby("bruit").size()))

v0 = pd.read_csv(ROOT / "results/resultats_jalon2.csv").assign(fichier=lambda d: d.file.str.replace(".csv", ""))
display(v0[["fichier", "single_train", "two_train"]].merge(
    res.loc[(res.type == "type_a") & (res.dossier == "cr06_ca03"), ["fichier", "unique_expl", "aleatoire_expl", "deux_expl"]],
    on="fichier").round(3))

## 7. Conclusions

*(à compléter)*